# 3. Generate weed-candidate masks and coordinates

Detect tobacco, threshold green vegetation, exclude crop boxes, and find remaining contour centers. These are candidate pixel coordinates, not verified weeds or calibrated treatment points. Processing is opt-in; notebook 4 evaluates saved predictions independently.


In [ ]:
from pathlib import Path
import sys
import os
import json
import tempfile

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks').is_dir() and (p / 'experiments/splits/items.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or a subdirectory.')
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import new_run, capture_environment
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'tobacco-notebook-mpl'))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'experiments/.ultralytics'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image


## Configure processing

Use trained single-class tobacco weights from notebook 2 or the project owner. Configuration is shared by preview, batch generation, and exported metadata.


In [ ]:
RUN_PROCESSING = False
SHOW_PREVIEW = True
IMAGES = ROOT / 'data/mask_validation/images'
WEIGHTS = ROOT / 'experiments/reproductions/detector-reproduction-01/yolo/yolov8x/weights/best.pt'
RUN_NAME = 'mask-reproduction-01'
PROCESSING_OPTIONS = dict(
    lower=(35, 40, 40), upper=(85, 255, 255), min_area=50.0, max_area=0.0,
    box_padding=0, conf=0.25, imgsz=1024, device='0',
)
mask_run = None
processing_complete = False


## See the historical pipeline

This is an existing qualitative panel, not a new model output or a ground-truth mask. The recorded processing used the tobacco image directory; that does not establish a separate weed validation set. When evaluating future masks, document which images were manually annotated and how they were held out from threshold tuning.


In [ ]:
display(Image(filename=str(ROOT / 'figures/publication/mask_pipeline_clean.jpg'), width=950))


## Load images and a trained detector

Optional dependencies are imported only when functions run. Empty or unreadable image inputs stop processing.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
VARIANTS = ["hsv_only", "crop_exclusion_only", "hsv_crop_exclusion", "final_contour_filtering"]

def list_images(directory: Path) -> list[Path]:
    if not directory.is_dir():
        raise ValueError(f"Missing image directory: {directory}")
    images = sorted(path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    if not images:
        raise ValueError(f"No images found in {directory}")
    stems = {}
    for path in images:
        if path.stem in stems:
            raise ValueError(f"Duplicate image stem: {stems[path.stem]} and {path}")
        stems[path.stem] = path
    return images

def load_detector(model_path: Path | None):
    if model_path is None:
        return None
    Path(os.environ["YOLO_CONFIG_DIR"]).mkdir(parents=True, exist_ok=True)
    from ultralytics import YOLO

    return YOLO(str(model_path))


## Find green vegetation

OpenCV reads BGR images; HSV thresholding uses hue 0–179. Green pixels can include tobacco and other vegetation.


In [ ]:
def hsv_mask(image, lower=(35, 40, 40), upper=(85, 255, 255)):
    import cv2
    import numpy as np
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, np.array(lower, dtype=np.uint8), np.array(upper, dtype=np.uint8))


### Synthetic example: find the green regions

These rectangles are software demonstrations, not field observations. The left green region represents a crop, the right one a weed candidate, and the tiny spot tests contour filtering.


In [ ]:
import cv2
import numpy as np
example_image = np.zeros((80, 120, 3), dtype=np.uint8)
example_image[10:50, 10:45] = (0, 255, 0)
example_image[20:60, 75:105] = (0, 255, 0)
example_image[65:68, 60:63] = (0, 255, 0)
example_vegetation = hsv_mask(example_image, lower=PROCESSING_OPTIONS['lower'], upper=PROCESSING_OPTIONS['upper'])
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(cv2.cvtColor(example_image, cv2.COLOR_BGR2RGB))
axes[0].set_title('Synthetic input')
axes[1].imshow(example_vegetation, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('HSV vegetation')
plt.show()


## Rasterize crop detections

Round and clip tobacco boxes to image bounds, then fill their rectangles. Their inverse also contains soil and is only a diagnostic.


In [ ]:
def rasterize_boxes(boxes, shape: tuple[int, int], *, box_padding: int = 0):
    """Round and clip xyxy boxes into a uint8 crop mask in original pixels."""
    import cv2
    import numpy as np
    mask = np.zeros(shape, dtype=np.uint8)
    height, width = shape
    for xyxy in boxes:
        x1, y1, x2, y2 = [int(round(value)) for value in xyxy[:4]]
        x1, y1 = max(0, x1 - box_padding), max(0, y1 - box_padding)
        x2, y2 = min(width - 1, x2 + box_padding), min(height - 1, y2 + box_padding)
        cv2.rectangle(mask, (x1, y1), (x2, y2), 255, thickness=-1)
    return mask

def crop_mask(detector, image_path: Path, shape: tuple[int, int], *,
              conf: float, imgsz: int, device: str, box_padding: int):
    """Run tobacco inference, then rasterize detections at the original size."""
    boxes = []
    if detector is not None:
        results = detector.predict(source=str(image_path), imgsz=imgsz, conf=conf,
                                   device=device, verbose=False)
        for result in results:
            if result.boxes is not None:
                boxes.extend(result.boxes.xyxy.cpu().numpy())
    return rasterize_boxes(boxes, shape, box_padding=box_padding)


In [ ]:
example_crops = rasterize_boxes([(8, 8, 47, 52)], example_vegetation.shape,
                               box_padding=PROCESSING_OPTIONS['box_padding'])
example_residual = cv2.bitwise_and(example_vegetation, cv2.bitwise_not(example_crops))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, mask, title in zip(axes, [example_crops, example_residual], ['Synthetic crop box', 'Vegetation minus crop']):
    ax.imshow(mask, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
plt.show()


## Filter contours and compute centers

Area means OpenCV contour area, not pixel count. External contours are filled, including holes. Moment centroids are truncated to integer coordinates; they need not lie on vegetation. A maximum area of zero disables that bound.


In [ ]:
def contour_filter(mask, min_area=50.0, max_area=0.0):
    import cv2
    import numpy as np
    filtered = np.zeros_like(mask)
    centers: list[tuple[int, int]] = []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        if max_area and area > max_area:
            continue
        cv2.drawContours(filtered, [contour], -1, 255, thickness=-1)
        moments = cv2.moments(contour)
        if moments["m00"]:
            centers.append((int(moments["m10"] / moments["m00"]), int(moments["m01"] / moments["m00"])))
    return filtered, centers


In [ ]:
example_final, example_centers = contour_filter(example_residual,
    min_area=PROCESSING_OPTIONS['min_area'], max_area=PROCESSING_OPTIONS['max_area'])
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(example_final, cmap='gray', vmin=0, vmax=255)
if example_centers:
    ax.scatter(*zip(*example_centers), color='red', marker='x')
ax.set_title('Synthetic filtered regions and centers')
plt.show()
display(pd.DataFrame(example_centers, columns=['x', 'y']))


The small spot is removed by the area bound. The remaining center describes contour geometry; it is not a calibrated treatment target.


## Compose the image pipeline

Preview and batch generation call this same function. Subtract crop regions from vegetation before filtering.


In [ ]:
def process_image(image_path: Path, detector, *, lower, upper, min_area: float,
                  max_area: float, box_padding: int, conf: float, imgsz: int, device: str) -> dict:
    """Return the original image, crop mask, named variants and pixel centers."""
    import cv2
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Cannot read {image_path}')
    vegetation = hsv_mask(image, lower, upper)
    crops = crop_mask(detector, image_path, vegetation.shape, conf=conf, imgsz=imgsz,
                      device=device, box_padding=box_padding)
    inverse = cv2.bitwise_not(crops)
    residual = cv2.bitwise_and(vegetation, inverse)
    final, centers = contour_filter(residual, min_area=min_area, max_area=max_area)
    variants = dict(zip(VARIANTS, [vegetation, inverse, residual, final]))
    return dict(image=image, crop_mask=crops, variants=variants, centers=centers)


## Initialize external processing

Validate inputs and load the detector once. Batch generation depends on this cell, not on the optional preview.


In [ ]:
processing_complete = False
detector = None
if RUN_PROCESSING:
    if not WEIGHTS.is_file():
        raise FileNotFoundError('Supply trained tobacco weights.')
    image_paths = list_images(IMAGES)
    detector = load_detector(WEIGHTS)
    if len(detector.names) != 1 or str(detector.names[0]).lower() != 'tobacco':
        raise ValueError(f'Expected one tobacco class: {detector.names}')
else:
    print('External processing disabled; synthetic examples above remain available.')


## Inspect one image step by step

Coordinates use the original image size, x rightward and y downward. The detector stays loaded for batch processing.


In [ ]:
if RUN_PROCESSING and SHOW_PREVIEW:
    import cv2
    preview = process_image(image_paths[0], detector, **PROCESSING_OPTIONS)
    bgr, crops = preview['image'], preview['crop_mask']
    variants, centers = preview['variants'], preview['centers']
    panels = [('Input RGB', cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)),
              ('HSV vegetation', variants['hsv_only']), ('Crop boxes', crops),
              ('Inverse crops (diagnostic)', variants['crop_exclusion_only']),
              ('Vegetation minus crops', variants['hsv_crop_exclusion']),
              ('Filtered mask', variants['final_contour_filtering'])]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout='constrained')
    for ax, (title, array) in zip(axes.flat, panels):
        ax.imshow(array, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    if centers:
        x, y = zip(*centers)
        axes.flat[-1].scatter(x, y, color='red', s=15)
    plt.show()
    display(pd.DataFrame(centers, columns=['x', 'y']))
else:
    print('External preview skipped.')


## Save batch predictions

Write four variants and header-preserving center CSVs. The configuration records the exact processing settings. A fresh parent run prevents stale files from mixing with predictions.


In [ ]:
def generate_predictions(images: Path, detector, output: Path, *, model_path: Path,
                         lower, upper, min_area: float, max_area: float, box_padding: int,
                         conf: float, imgsz: int, device: str) -> int:
    """Export named masks, center CSVs and the exact processing settings."""
    import cv2
    import csv
    paths = list_images(images)
    for variant in [*VARIANTS, 'centers']:
        (output / variant).mkdir(parents=True, exist_ok=False)
    for path in paths:
        result = process_image(path, detector, lower=lower, upper=upper,
            min_area=min_area, max_area=max_area, box_padding=box_padding,
            conf=conf, imgsz=imgsz, device=device)
        variants, centers = result['variants'], result['centers']
        for name, mask in variants.items():
            target = output / name / f'{path.stem}.png'
            if not cv2.imwrite(str(target), mask):
                raise OSError(f'Cannot write {target}')
        with (output / 'centers' / f'{path.stem}.csv').open('w', newline='', encoding='utf-8') as handle:
            writer = csv.writer(handle)
            writer.writerow(['x', 'y'])
            writer.writerows(centers)
    config = dict(images=str(images), model=str(model_path) if model_path else None,
        device=device, conf=conf, imgsz=imgsz,
        hsv={'h': [lower[0], upper[0]], 's': [lower[1], upper[1]], 'v': [lower[2], upper[2]]},
        min_area=min_area, max_area=max_area, box_padding=box_padding, variants=VARIANTS)
    (output / 'mask_config.json').write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
    return len(paths)


In [ ]:
processing_complete = False
if RUN_PROCESSING:
    mask_run = new_run(ROOT, RUN_NAME)
    capture_environment(mask_run / 'environment')
    count = generate_predictions(IMAGES.resolve(), detector, mask_run / 'masks',
                                 model_path=WEIGHTS.resolve(), **PROCESSING_OPTIONS)
    processing_complete = True
    print(f'Processed {count} images. Set PREDICTIONS in notebook 4 to:', mask_run / 'masks')
else:
    print('Batch processing skipped.')


## Interpret the outputs responsibly

Crop boxes can remove nearby weeds, and missed crops can become false weed candidates. Illumination affects fixed HSV thresholds. Contour centroids are geometric summaries rather than verified treatment points. Keep new prediction files, manual annotations, and historical evidence distinct.

The present article supports quantitative tobacco detection and qualitative mask/coordinate demonstrations. New manual annotations could support additional evaluation, but no notebook output alone validates physical laser treatment. Record dataset provenance, annotation policy, held-out image selection, settings, and hardware with any new results.
